# Computer Vision Nanodegree

## Project: Image Captioning

---

In this notebook, you will train your CNN-RNN model.  

You are welcome and encouraged to try out many different architectures and hyperparameters when searching for a good model.

This does have the potential to make the project quite messy!  Before submitting your project, make sure that you clean up:
- the code you write in this notebook.  The notebook should describe how to train a single CNN-RNN architecture, corresponding to your final choice of hyperparameters.  You should structure the notebook so that the reviewer can replicate your results by running the code in this notebook.  
- the output of the code cell in **Step 2**.  The output should show the output obtained when training the model from scratch.

This notebook **will be graded**.  

Feel free to use the links below to navigate the notebook:
- [Step 1](#step1): Training Setup
- [Step 2](#step2): Train your Model
- [Step 3](#step3): (Optional) Validate your Model

<a id='step1'></a>
## Step 1: Training Setup

In this step of the notebook, you will customize the training of your CNN-RNN model by specifying hyperparameters and setting other options that are important to the training procedure.  The values you set now will be used when training your model in **Step 2** below.

You should only amend blocks of code that are preceded by a `TODO` statement.  **Any code blocks that are not preceded by a `TODO` statement should not be modified**.

### Task #1

Begin by setting the following variables:
- `batch_size` - the batch size of each training batch.  It is the number of image-caption pairs used to amend the model weights in each training step. 
- `vocab_threshold` - the minimum word count threshold.  Note that a larger threshold will result in a smaller vocabulary, whereas a smaller threshold will include rarer words and result in a larger vocabulary.  
- `vocab_from_file` - a Boolean that decides whether to load the vocabulary from file. 
- `embed_size` - the dimensionality of the image and word embeddings.  
- `hidden_size` - the number of features in the hidden state of the RNN decoder.  
- `num_epochs` - the number of epochs to train the model.  We recommend that you set `num_epochs=3`, but feel free to increase or decrease this number as you wish.  [This paper](https://arxiv.org/pdf/1502.03044.pdf) trained a captioning model on a single state-of-the-art GPU for 3 days, but you'll soon see that you can get reasonable results in a matter of a few hours!  (_But of course, if you want your model to compete with current research, you will have to train for much longer._)
- `save_every` - determines how often to save the model weights.  We recommend that you set `save_every=1`, to save the model weights after each epoch.  This way, after the `i`th epoch, the encoder and decoder weights will be saved in the `models/` folder as `encoder-i.pkl` and `decoder-i.pkl`, respectively.
- `print_every` - determines how often to print the batch loss to the Jupyter notebook while training.  Note that you **will not** observe a monotonic decrease in the loss function while training - this is perfectly fine and completely expected!  You are encouraged to keep this at its default value of `100` to avoid clogging the notebook, but feel free to change it.
- `log_file` - the name of the text file containing - for every step - how the loss and perplexity evolved during training.

If you're not sure where to begin to set some of the values above, you can peruse [this paper](https://arxiv.org/pdf/1502.03044.pdf) and [this paper](https://arxiv.org/pdf/1411.4555.pdf) for useful guidance!  **To avoid spending too long on this notebook**, you are encouraged to consult these suggested research papers to obtain a strong initial guess for which hyperparameters are likely to work best.  Then, train a single model, and proceed to the next notebook (**3_Inference.ipynb**).  If you are unhappy with your performance, you can return to this notebook to tweak the hyperparameters (and/or the architecture in **model.py**) and re-train your model.

### Question 1

**Question:** Describe your CNN-RNN architecture in detail.  With this architecture in mind, how did you select the values of the variables in Task 1?  If you consulted a research paper detailing a successful implementation of an image captioning model, please provide the reference.

**Answer:** The architecture follows the encoder-decoder design from *Show and Tell* (Vinyals et al., 2015, https://arxiv.org/pdf/1411.4555.pdf). The **CNNEncoder** is a pretrained ResNet-50 with its final classification layer removed; the resulting 2048-d pooled feature vector is passed through a trainable `nn.Linear` layer that projects it into the same `embed_size`-dimensional space used for word embeddings, so the image can be treated as the first "word" in the caption sequence. The **RNNDecoder** embeds each caption token to `embed_size`, concatenates the image feature as the first timestep, and feeds the sequence through a single-layer LSTM with `hidden_size` hidden units, followed by a linear layer projecting each hidden state to a distribution over the vocabulary.

Hyperparameters were chosen to match the values suggested by the Show and Tell paper and the common baseline used for this project: `embed_size=256` and `hidden_size=512` give the LSTM enough capacity to model caption structure without being slow to train on a single GPU. `batch_size=32` fits comfortably in the memory of a single Tesla T4 GPU, gives frequent weight updates, and keeps gradient estimates reasonably stable. `vocab_threshold=5` follows the paper's convention of discarding very rare words, which keeps the vocabulary (and therefore the final linear layer) a manageable size without losing much expressiveness. `num_epochs=3` was kept at the recommended value since it is enough to see the loss drop substantially in the time available.


### (Optional) Task #2

Note that we have provided a recommended image transform `transform_train` for pre-processing the training images, but you are welcome (and encouraged!) to modify it as you wish.  When modifying this transform, keep in mind that:
- the images in the dataset have varying heights and widths, and 
- if using a pre-trained model, you must perform the corresponding appropriate normalization.

### Question 2

**Question:** How did you select the transform in `transform_train`?  If you left the transform at its provided value, why do you think that it is a good choice for your CNN architecture?

**Answer:** The transform was left at its provided value. It resizes the smaller edge to 256px and takes a random 224x224 crop with a random horizontal flip, which is the standard input size expected by ResNet-50 and provides useful data augmentation during training. The final `ToTensor` + `Normalize` step uses the ImageNet channel mean/std that ResNet-50 was pretrained with, which is required for the frozen convolutional weights in `CNNEncoder` to produce meaningful features. Since the encoder architecture wasn't changed, this transform stays congruent with it, and it is reused unchanged (without the random crop/flip) as `transform_test` in **3_Inference.ipynb**.

### Task #3

Next, you will specify a Python list containing the learnable parameters of the model.  For instance, if you decide to make all weights in the decoder trainable, but only want to train the weights in the embedding layer of the encoder, then you should set `params` to something like:
```
params = list(decoder.parameters()) + list(encoder.embed.parameters()) 
```

### Question 3

**Question:** How did you select the trainable parameters of your architecture?  Why do you think this is a good choice?

**Answer:** All of `decoder.parameters()` are trainable, since the RNNDecoder is initialized from scratch and needs to learn both the word embeddings and the language model from scratch. For the encoder, the ResNet-50 backbone is already pretrained on ImageNet and frozen (`requires_grad_(False)` in `CNNEncoder.__init__`) to avoid overfitting it to the much smaller COCO caption dataset and to keep training fast; only `encoder.embed`, the new linear layer that maps ResNet features into the shared embedding space, is trainable, since it has no pretrained weights and must be learned to align image features with the word embedding space used by the decoder.

### Task #4

Finally, you will select an [optimizer](http://pytorch.org/docs/master/optim.html#torch.optim.Optimizer).

### Question 4

**Question:** How did you select the optimizer used to train your model?

**Answer:** Adam was chosen as the optimizer (`torch.optim.Adam(params, lr=0.001)`) because it combines momentum with per-parameter adaptive learning rates, which tends to converge faster and more reliably than plain SGD on recurrent language models like the LSTM decoder used here, without requiring extensive manual learning rate tuning. The default learning rate of 0.001 is a standard, well-tested starting point for Adam and worked well in early loss curves.


In [ ]:
import nltk
nltk.download('punkt')

In [ ]:
# --- COCO dataset auto-setup (added) ---
# Ensures the COCO annotations/images this notebook needs exist somewhere writable.
# Tries /opt first (the default `cocoapi_loc` expected by data_loader.py); falls back
# to the home directory or the current directory if /opt isn't writable in this
# workspace. Safe to re-run: anything already downloaded/extracted is skipped.
import os
import urllib.request
import zipfile

def _writable(path):
    try:
        os.makedirs(path, exist_ok=True)
        probe = os.path.join(path, '.write_test')
        with open(probe, 'w') as f:
            f.write('ok')
        os.remove(probe)
        return True
    except OSError:
        return False

COCOAPI_LOC = None
for _root in ['/opt', os.path.expanduser('~'), '/workspace', '.']:
    if _writable(os.path.join(_root, 'cocoapi')):
        COCOAPI_LOC = _root
        break
if COCOAPI_LOC is None:
    raise RuntimeError('Could not find a writable location to store the COCO dataset.')
print('Using COCOAPI_LOC =', COCOAPI_LOC)

_coco_root = os.path.join(COCOAPI_LOC, 'cocoapi')
os.makedirs(os.path.join(_coco_root, 'annotations'), exist_ok=True)
os.makedirs(os.path.join(_coco_root, 'images'), exist_ok=True)

def _download(url, dest):
    if os.path.exists(dest):
        print('Already downloaded:', dest)
        return
    print('Downloading', url, '-> ', dest)
    urllib.request.urlretrieve(url, dest)

def _extract(zip_path, dest_dir, done_marker):
    if os.path.exists(done_marker):
        print('Already extracted:', done_marker)
        return
    print('Extracting', zip_path, '-> ', dest_dir)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(dest_dir)

# Captions + instances annotations (~241 MB) -- covers train2014 and val2014.
_ann_zip = os.path.join(_coco_root, 'annotations_trainval2014.zip')
_download('http://images.cocodataset.org/annotations/annotations_trainval2014.zip', _ann_zip)
_extract(_ann_zip, _coco_root, os.path.join(_coco_root, 'annotations', 'captions_train2014.json'))

# Training images (~13 GB).
_train_zip = os.path.join(_coco_root, 'train2014.zip')
_download('http://images.cocodataset.org/zips/train2014.zip', _train_zip)
_extract(_train_zip, os.path.join(_coco_root, 'images'), os.path.join(_coco_root, 'images', 'train2014'))

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
import sys
sys.path.append('/opt/cocoapi/PythonAPI')
from pycocotools.coco import COCO
from data_loader import get_loader
from model import EncoderCNN, DecoderRNN
import math


## TODO #1: Select appropriate values for the Python variables below.
batch_size = 32           # batch size
vocab_threshold = 5        # minimum word count threshold
vocab_from_file = False    # if True, load existing vocab file
embed_size = 256           # dimensionality of image and word embeddings
hidden_size = 512          # number of features in hidden state of the RNN decoder
num_epochs = 3             # number of training epochs
save_every = 1             # determines frequency of saving model weights
print_every = 100          # determines window for printing average loss
log_file = 'training_log.txt'       # name of file with saved training loss and perplexity

# (Optional) TODO #2: Amend the image transform below.
transform_train = transforms.Compose([ 
    transforms.Resize(256),                          # smaller edge of image resized to 256
    transforms.RandomCrop(224),                      # get 224x224 crop from random location
    transforms.RandomHorizontalFlip(),               # horizontally flip image with probability=0.5
    transforms.ToTensor(),                           # convert the PIL Image to a tensor
    transforms.Normalize((0.485, 0.456, 0.406),      # normalize image for pre-trained model
                         (0.229, 0.224, 0.225))])

# Build data loader.
data_loader = get_loader(transform=transform_train,
                         mode='train',
                         batch_size=batch_size,
                         vocab_threshold=vocab_threshold,
                         vocab_from_file=vocab_from_file,
                         cocoapi_loc=COCOAPI_LOC)

# The size of the vocabulary.
vocab_size = len(data_loader.dataset.vocab)

# Initialize the encoder and decoder. 
encoder = EncoderCNN(embed_size)
decoder = DecoderRNN(embed_size, hidden_size, vocab_size)

# Move models to GPU if CUDA is available. 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder.to(device)
decoder.to(device)

# Define the loss function. 
criterion = nn.CrossEntropyLoss().cuda() if torch.cuda.is_available() else nn.CrossEntropyLoss()

# TODO #3: Specify the learnable parameters of the model.
# The encoder's ResNet-50 weights are frozen (see model.py), so only its final
# embedding layer is trainable; the decoder is trained in full.
params = list(decoder.parameters()) + list(encoder.embed.parameters())

# TODO #4: Define the optimizer.
optimizer = torch.optim.Adam(params, lr=0.001)

# Set the total number of training steps per epoch.
total_step = math.ceil(len(data_loader.dataset.caption_lengths) / data_loader.batch_sampler.batch_size)


<a id='step2'></a>
## Step 2: Train your Model

Once you have executed the code cell in **Step 1**, the training procedure below should run without issue.  

It is completely fine to leave the code cell below as-is without modifications to train your model.  However, if you would like to modify the code used to train the model below, you must ensure that your changes are easily parsed by your reviewer.  In other words, make sure to provide appropriate comments to describe how your code works!  

You may find it useful to load saved weights to resume training.  In that case, note the names of the files containing the encoder and decoder weights that you'd like to load (`encoder_file` and `decoder_file`).  Then you can load the weights by using the lines below:

```python
# Load pre-trained weights before resuming training.
encoder.load_state_dict(torch.load(os.path.join('./models', encoder_file)))
decoder.load_state_dict(torch.load(os.path.join('./models', decoder_file)))
```

While trying out parameters, make sure to take extensive notes and record the settings that you used in your various training runs.  In particular, you don't want to encounter a situation where you've trained a model for several hours but can't remember what settings you used :).

### A Note on Tuning Hyperparameters

To figure out how well your model is doing, you can look at how the training loss and perplexity evolve during training - and for the purposes of this project, you are encouraged to amend the hyperparameters based on this information.  

However, this will not tell you if your model is overfitting to the training data, and, unfortunately, overfitting is a problem that is commonly encountered when training image captioning models.  

For this project, you need not worry about overfitting. **This project does not have strict requirements regarding the performance of your model**, and you just need to demonstrate that your model has learned **_something_** when you generate captions on the test data.  For now, we strongly encourage you to train your model for the suggested 3 epochs without worrying about performance; then, you should immediately transition to the next notebook in the sequence (**3_Inference.ipynb**) to see how your model performs on the test data.  If your model needs to be changed, you can come back to this notebook, amend hyperparameters (if necessary), and re-train the model.

That said, if you would like to go above and beyond in this project, you can read about some approaches to minimizing overfitting in section 4.3.1 of [this paper](http://ieeexplore.ieee.org/stamp/stamp.jsp?arnumber=7505636).  In the next (optional) step of this notebook, we provide some guidance for assessing the performance on the validation dataset.

In [ ]:
import torch.utils.data as data
import numpy as np
import os
import glob
import time
import torch

# Limit PyTorch to one CPU thread. In this shared container the default (16 threads)
# oversubscribes the CPU and made image pre-processing ~15x slower (168 ms vs 11 ms
# per image), which left the GPU idle for most of each training step.
torch.set_num_threads(1)

os.makedirs('./models', exist_ok=True)

# ---- Resume support (the workspace can shut down on inactivity) ----
# resume=True continues from the newest saved state, if any exists; otherwise training
# starts from scratch. Set resume=False to force a fresh run.
resume = True
checkpoint_every = 2000                  # also save a resumable checkpoint every N steps
checkpoint_path = './models/checkpoint.pt'
start_epoch, start_step = 1, 1
resumed = False

if resume and os.path.exists(checkpoint_path):
    # Mid-epoch checkpoint: model weights, optimizer state and the position reached.
    ckpt = torch.load(checkpoint_path, map_location=device)
    encoder.load_state_dict(ckpt['encoder'])
    decoder.load_state_dict(ckpt['decoder'])
    optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch, start_step = ckpt['epoch'], ckpt['i_step'] + 1
    resumed = True
elif resume:
    # No checkpoint: fall back to the newest end-of-epoch weights (encoder-N.pkl / decoder-N.pkl).
    done = [int(p.split('-')[-1].split('.')[0]) for p in glob.glob('./models/encoder-*.pkl')
            if os.path.exists(p.replace('encoder-', 'decoder-'))]
    if done:
        last = max(done)
        encoder.load_state_dict(torch.load(os.path.join('./models', 'encoder-%d.pkl' % last), map_location=device))
        decoder.load_state_dict(torch.load(os.path.join('./models', 'decoder-%d.pkl' % last), map_location=device))
        start_epoch = last + 1
        resumed = True

if resumed:
    print('Resuming from epoch %d, step %d' % (start_epoch, start_step))

def save_checkpoint(epoch, i_step):
    torch.save({'encoder': encoder.state_dict(), 'decoder': decoder.state_dict(),
                'optimizer': optimizer.state_dict(), 'epoch': epoch, 'i_step': i_step}, checkpoint_path)

# Open the training log file (append when resuming so earlier progress is kept).
f = open(log_file, 'a' if resumed else 'w')

for epoch in range(start_epoch, num_epochs+1):
    
    first_step = start_step if epoch == start_epoch else 1
    for i_step in range(first_step, total_step+1):
        
        # Randomly sample a caption length, and sample indices with that length.
        indices = data_loader.dataset.get_train_indices()
        # Create and assign a batch sampler to retrieve a batch with the sampled indices.
        new_sampler = data.sampler.SubsetRandomSampler(indices=indices)
        data_loader.batch_sampler.sampler = new_sampler
        
        # Obtain the batch.
        images, captions = next(iter(data_loader))

        # Move batch of images and captions to GPU if CUDA is available.
        images = images.to(device)
        captions = captions.to(device)
        
        # Zero the gradients.
        decoder.zero_grad()
        encoder.zero_grad()
        
        # Pass the inputs through the CNN-RNN model.
        features = encoder(images)
        outputs = decoder(features, captions)
        
        # Calculate the batch loss.
        loss = criterion(outputs.view(-1, vocab_size), captions.view(-1))
        
        # Backward pass.
        loss.backward()
        
        # Update the parameters in the optimizer.
        optimizer.step()
            
        # Get training statistics.
        stats = 'Epoch [%d/%d], Step [%d/%d], Loss: %.4f, Perplexity: %5.4f' % (epoch, num_epochs, i_step, total_step, loss.item(), np.exp(loss.item()))
        
        # Print training statistics to file.
        f.write(stats + '\n')
        f.flush()
        
        # Print training statistics to the notebook only every `print_every` steps.
        # (Printing on every single step floods the notebook's websocket connection
        # over a run of thousands of steps, which can cause the remote kernel/server
        # connection to drop -- so we deliberately do not print each step.)
        if i_step % print_every == 0:
            print('\r' + stats)
        
        # Save a resumable checkpoint periodically.
        if i_step % checkpoint_every == 0:
            save_checkpoint(epoch, i_step)
            
    # Save the weights.
    if epoch % save_every == 0:
        torch.save(decoder.state_dict(), os.path.join('./models', 'decoder-%d.pkl' % epoch))
        torch.save(encoder.state_dict(), os.path.join('./models', 'encoder-%d.pkl' % epoch))
    save_checkpoint(epoch, total_step)      # end-of-epoch position, so a resume starts the next epoch

# Close the training log file.
f.close()


<a id='step3'></a>
## Step 3: (Optional) Validate your Model

To assess potential overfitting, one approach is to assess performance on a validation set.  If you decide to do this **optional** task, you are required to first complete all of the steps in the next notebook in the sequence (**3_Inference.ipynb**); as part of that notebook, you will write and test code (specifically, the `sample` method in the `DecoderRNN` class) that uses your RNN decoder to generate captions.  That code will prove incredibly useful here. 

If you decide to validate your model, please do not edit the data loader in **data_loader.py**.  Instead, create a new file named **data_loader_val.py** containing the code for obtaining the data loader for the validation data.  You can access:
- the validation images at filepath `'/opt/cocoapi/images/train2014/'`, and
- the validation image caption annotation file at filepath `'/opt/cocoapi/annotations/captions_val2014.json'`.

The suggested approach to validating your model involves creating a json file such as [this one](https://github.com/cocodataset/cocoapi/blob/master/results/captions_val2014_fakecap_results.json) containing your model's predicted captions for the validation images.  Then, you can write your own script or use one that you [find online](https://github.com/tylin/coco-caption) to calculate the BLEU score of your model.  You can read more about the BLEU score, along with other evaluation metrics (such as TEOR and Cider) in section 4.1 of [this paper](https://arxiv.org/pdf/1411.4555.pdf).  For more information about how to use the annotation file, check out the [website](http://cocodataset.org/#download) for the COCO dataset.

### Validation approach

The cells below generate a caption for a random subset of the COCO **val2014** images (images the model never trained on) with the `sample` method of `DecoderRNN`, write the predictions to a COCO-style results file (`captions_val2014_results.json`), and score them against the five human reference captions per image using **BLEU-1 to BLEU-4** (`nltk`). The validation data loader lives in **data_loader_val.py**; **data_loader.py** is untouched.

Run the cell in **Step 2** first (or load saved weights: see the note in Step 2), so that `encoder`, `decoder` and `data_loader` exist.

In [ ]:
# Download the validation images if needed, then build the validation data loader.
# (_download, _extract, _coco_root and COCOAPI_LOC come from the data set-up cell in Step 1.)
_val_zip = os.path.join(_coco_root, 'val2014.zip')
_download('http://images.cocodataset.org/zips/val2014.zip', _val_zip)   # ~6.2 GB, skipped if present
_extract(_val_zip, os.path.join(_coco_root, 'images'), os.path.join(_coco_root, 'images', 'val2014'))

from data_loader_val import get_loader_val

# Same pre-processing as the test images (no random crop / flip), matching transform_train's normalization.
transform_val = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406),
                         (0.229, 0.224, 0.225))])

val_size = 1000    # number of validation images to score; set to None to use all ~40k
val_loader = get_loader_val(transform=transform_val, cocoapi_loc=COCOAPI_LOC, subset_size=val_size)
val_dataset = val_loader.dataset
print('validation images:', len(val_dataset))

In [ ]:
import json
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

# Optional: score a saved checkpoint instead of the weights currently in memory.
# encoder.load_state_dict(torch.load(os.path.join('./models', 'encoder-3.pkl')))
# decoder.load_state_dict(torch.load(os.path.join('./models', 'decoder-3.pkl')))

encoder.eval()
decoder.eval()
vocab = data_loader.dataset.vocab

results = []       # COCO-format predictions: [{'image_id': ..., 'caption': ...}, ...]
hypotheses = []    # predicted captions as token lists
references = []    # for each image, the list of tokenized reference captions

with torch.no_grad():
    for image, img_id in val_loader:
        img_id = img_id.item()

        # Generate a caption for this image.
        features = encoder(image.to(device)).unsqueeze(1)
        output = decoder.sample(features)

        # Convert predicted indices to words, dropping <start> and stopping at <end>.
        words = []
        for idx in output:
            word = vocab.idx2word[idx]
            if word == vocab.end_word:
                break
            if word != vocab.start_word:
                words.append(word)

        results.append({'image_id': img_id, 'caption': ' '.join(words)})
        hypotheses.append(words)
        # Tokenize references exactly as the training captions were tokenized.
        references.append([nltk.tokenize.word_tokenize(ref.lower())
                           for ref in val_dataset.references(img_id)])

# Save predictions in the format used by the COCO caption evaluation tools.
with open('captions_val2014_results.json', 'w') as f:
    json.dump(results, f)

# BLEU-1 ... BLEU-4 (smoothing avoids zero scores on short captions).
smooth = SmoothingFunction().method1
for n in range(1, 5):
    weights = tuple(1.0 / n if i < n else 0.0 for i in range(4))
    score = corpus_bleu(references, hypotheses, weights=weights, smoothing_function=smooth)
    print('BLEU-%d: %.4f' % (n, score))

# A few example predictions next to one reference caption.
for r, refs in list(zip(results, references))[:5]:
    print('\nimage %d\n  predicted: %s\n  reference: %s' % (r['image_id'], r['caption'], ' '.join(refs[0])))

encoder.train()
decoder.train()